In [ ]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
import torch, rdkit
import os
from pathlib import Path
import sys
def _is_models_root(p: Path) -> bool:
    return (p / "utils").is_dir() and (p / "models").is_dir() and (p / "notebooks").is_dir() and (p / "data").is_dir()


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if _is_models_root(p):
            return p
        cand = p / "MY_PAPER_RELATED" / "MODELS"
        if _is_models_root(cand):
            return cand
    raise FileNotFoundError("Could not locate MODELS project root")
PROJECT_ROOT = resolve_project_root()
os.environ["MODELS_VARIANT"] = "TransCVAE"
os.environ["MODELS_FULL_DATA_TRAINING"] = "0"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rdkit import Chem
from utils.utils import *
from utils.dataloader import *
from utils.loss_fn import *
from utils.eval import *

from tqdm import trange
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test = sfp.encoder_psmiles("CC(CCCNC(=O)C(C)OC(=O)[*])O[*]")
print(test)
print(list(sfp.split_selfies(test)))
vocab = dataset.vocab
index_to_token = {idx: token for token, idx in vocab.items()}
print("Split summary:")
print(split_summary.to_string(index=False))
print("Split leakage counts:", split_info["leak_counts"])
print(f"Split tag: {SPLIT_TAG}")
print(f"Full-data training mode: {FULL_DATA_TRAINING}")
print(f"Dataset sizes - train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")


In [ ]:
def select_model(latent):
    from models.Trans import CVAE, PriorNet
    model = CVAE(latent_dim=latent).to(device)
    prior = PriorNet(y_dim=1, latent_dim=latent).to(device)
    return model, prior


def get_loss_fn(latent):
    # Quality-first rebalance for TransCVAE: keep diversity, recover BCE/Tanimoto.
    loss_fn = ConditionalVAELoss(
        vocab_size=dataset.vocab_size,
        max_beta=0.66,
        cyc_steps=1200,
        num_cycles=1,
        anneal_steps=600,
        free_bits=0.05,
        capacity_max=0.0,
        capacity_inc=0.0,
        gamma=0.8,
        prop_w=0.12,
        nce=0.001,
        sig_pen_p=0.0015,
        sig_pen_q=0.0020,
        imb=0.04,
        mu_align_w=0.0005,
        sigma_p_target=0.215,
        sigma_q_target=0.20,
        sigma_p_target_w=0.11,
        sigma_q_target_w=0.04,
        sigma_align_w=0.016,
        sigma_p_var_floor=0.050,
        sigma_p_var_w=1.10,
        sigma_q_var_floor=0.018,
        sigma_q_var_w=0.10,
        sigma_p_upper_margin=0.15,
        sigma_p_upper_w=0.10,
        sigma_p_hi_ratio_target=0.50,
        sigma_p_hi_ratio_tau=0.03,
        sigma_p_hi_ratio_w=0.03,
        mu_std_floor=0.17,
        mu_std_w=0.32,
        mu_std_global_floor=0.20,
        mu_std_global_w=0.28,
        mu_energy_floor=0.040,
        mu_energy_w=0.24,
        reg_warmup_steps=700,
        latent_dim=latent,
    ).to(device)
    return loss_fn



In [ ]:
mode = "Trans"
latent_dim = 128
seq_len = dataset.max_len + 2

lr = 3e-4
lr_prior = lr * 0.1
from torch.optim import AdamW


def set_data_context(ctx):
    global dataset, train_dataset, val_dataset, test_dataset, trainval_dataset
    global train_dataloader, val_dataloader, test_dataloader, trainval_dataloader, eval_dataloader
    global train_indices, val_indices, test_indices, trainval_indices
    global fold_assignment, split_summary, split_info, SPLIT_TAG, vocab, index_to_token, PAD_IDX, seq_len

    dataset = ctx["dataset"]
    train_dataset = ctx["train_dataset"]
    val_dataset = ctx["val_dataset"]
    test_dataset = ctx["test_dataset"]
    trainval_dataset = ctx["trainval_dataset"]
    train_dataloader = ctx["train_dataloader"]
    val_dataloader = ctx["val_dataloader"]
    test_dataloader = ctx["test_dataloader"]
    trainval_dataloader = ctx["trainval_dataloader"]
    eval_dataloader = ctx["eval_dataloader"]
    train_indices = ctx["train_indices"]
    val_indices = ctx["val_indices"]
    test_indices = ctx["test_indices"]
    trainval_indices = ctx["trainval_indices"]
    fold_assignment = ctx["fold_assignment"]
    split_summary = ctx["split_summary"]
    split_info = ctx["split_info"]
    SPLIT_TAG = ctx["split_tag"]
    vocab = dataset.vocab
    index_to_token = {idx: token for token, idx in vocab.items()}
    PAD_IDX = dataset.vocab["[PAD]"]
    seq_len = dataset.max_len + 2
    return ctx


def init_model_bundle():
    model, prior = select_model(latent_dim)
    loss_fn = get_loss_fn(latent_dim)

    for head in [model.to_prop, model.to_prop_z]:
        nn.init.xavier_uniform_(head.weight)
        nn.init.zeros_(head.bias)

    prop_params = list(model.to_prop.parameters()) + list(model.to_prop_z.parameters())
    prop_params_id = {id(p) for p in prop_params}
    base_params = [p for p in model.parameters() if id(p) not in prop_params_id]

    optim = AdamW([
        {"params": base_params, "lr": lr},
        {"params": prop_params, "lr": lr * 0.5},
    ])
    optim2 = AdamW(prior.parameters(), lr=lr_prior)
    return model, prior, loss_fn, optim, optim2


model, prior, loss_fn, optim, optim2 = init_model_bundle()


In [ ]:
import copy
import datetime
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.error')

status_out = widgets.Output()
PAD_IDX = dataset.vocab['[PAD]']
display(status_out)

MAX_EPOCHS = 1200
VAL_EVERY = 10
EARLY_STOP_PATIENCE = 20
EARLY_STOP_MIN_DELTA = 1e-4
RUN_FULL_CV = True
RUN_FINAL_TRAIN_ON_TRAINVAL = True

VARIANT_NAME = os.environ.get("MODELS_VARIANT", mode)
CV_VAL_FOLDS = [f for f in range(SPLIT_N_FOLDS) if f != TEST_FOLD]
CV_CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "cv_es" / VARIANT_NAME
CV_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def _cvae_latent_mask(enc_mask):
    if mode == "LSTM":
        return torch.ones((enc_mask.size(0), 1), dtype=torch.bool, device=enc_mask.device)
    return torch.cat(
        ((~enc_mask), torch.ones(enc_mask.size(0), 1, dtype=torch.bool, device=enc_mask.device)),
        dim=1,
    )


def _cvae_forward_loss(model, prior, loss_fn, batch, step, train: bool):
    sm_enc, sm_dec_in, sm_dec_out, prop = [t.to(device) for t in batch]
    enc_mask = sm_enc == PAD_IDX
    dec_mask = sm_dec_in == PAD_IDX

    output, tgt, mu_q, lv_q, tgt_z = model(
        sm_enc,
        sm_dec_in,
        prop.float(),
        enc_mask,
        dec_mask,
    )
    mu_p, lv_p = prior(prop.float().squeeze(-1))
    latent_mask = _cvae_latent_mask(enc_mask)
    loss, BCE, KLD, kld_raw, kld_per_token, props = loss_fn(
        output.float(),
        sm_dec_out,
        mu_q,
        lv_q,
        mu_p,
        lv_p,
        tgt,
        prop.float().squeeze(-1),
        tgt_z,
        step,
        latent_mask=latent_mask,
    )

    with torch.no_grad():
        pred_tok = output.argmax(dim=-1)
        nonpad_mask = sm_dec_out != PAD_IDX
        token_total = int(nonpad_mask.sum().item())
        token_correct = int(((pred_tok == sm_dec_out) & nonpad_mask).sum().item())
        bce_total = float(BCE.item()) * max(1, sm_enc.size(0))
        sigma_q = torch.exp(0.5 * lv_q.detach())
        sigma_p = torch.exp(0.5 * lv_p.detach())

    stats = {
        "loss": float(loss.item()),
        "bce": float(BCE.item()),
        "kld": float(KLD.item()),
        "kld_raw": float(kld_raw.item()),
        "bce_total": bce_total,
        "token_total": token_total,
        "token_correct": token_correct,
        "mu_q": mu_q.detach().cpu(),
        "sigma_q": sigma_q.cpu(),
        "sigma_p": sigma_p.cpu(),
        "latent_mask": latent_mask.detach().cpu(),
    }
    return loss, stats


def _aggregate_cvae_stats(stats_list, n_batches):
    if not stats_list:
        return {"loss": float("nan"), "ce_per_token": float("nan"), "tok_acc": float("nan")}

    token_total = sum(s["token_total"] for s in stats_list)
    token_correct = sum(s["token_correct"] for s in stats_list)
    bce_total = sum(s["bce_total"] for s in stats_list)
    metrics = {
        "loss": sum(s["loss"] for s in stats_list) / max(1, n_batches),
        "bce_epoch": sum(s["bce"] for s in stats_list) / max(1, n_batches),
        "kld_epoch": sum(s["kld"] for s in stats_list) / max(1, n_batches),
        "kld_raw": sum(s["kld_raw"] for s in stats_list) / max(1, n_batches),
        "ce_per_token": bce_total / max(1, token_total),
        "tok_acc": token_correct / max(1, token_total),
    }

    mu_ext = torch.cat([s["mu_q"] for s in stats_list], dim=0)
    mask = torch.cat([s["latent_mask"] for s in stats_list], dim=0).bool()
    sigma_q_ext = torch.cat([s["sigma_q"] for s in stats_list], dim=0)
    sigma_p_ext = torch.cat([s["sigma_p"] for s in stats_list], dim=0)
    if mask.any() and mu_ext.dim() == 3:
        mu_valid = mu_ext[mask]
        sigma_q_valid = sigma_q_ext[mask]
        sigma_p_valid = sigma_p_ext[mask]
    else:
        mu_valid = mu_ext.reshape(-1, latent_dim)
        sigma_q_valid = sigma_q_ext.reshape(-1, latent_dim)
        sigma_p_valid = sigma_p_ext.reshape(-1, latent_dim)

    metrics.update({
        "mu_perdim_std": mu_valid.std(dim=0, unbiased=False).mean().item() if mu_valid.numel() else 0.0,
        "mu_std": mu_valid.std(unbiased=False).item() if mu_valid.numel() else 0.0,
        "sigma_q_mean": sigma_q_valid.mean().item() if sigma_q_valid.numel() else 0.0,
        "sigma_q_std": sigma_q_valid.std(unbiased=False).item() if sigma_q_valid.numel() > 1 else 0.0,
        "sigma_p_mean": sigma_p_valid.mean().item() if sigma_p_valid.numel() else 0.0,
        "sigma_p_std": sigma_p_valid.std(unbiased=False).item() if sigma_p_valid.numel() > 1 else 0.0,
    })
    return metrics


def train_one_epoch(model, prior, loss_fn, optim, optim2, loader, step):
    model.train()
    prior.train()
    stats_list = []
    for batch in loader:
        optim.zero_grad()
        optim2.zero_grad()
        loss, stats = _cvae_forward_loss(model, prior, loss_fn, batch, step, train=True)
        loss.backward()
        optim.step()
        optim2.step()
        stats_list.append(stats)
    return _aggregate_cvae_stats(stats_list, len(loader))


def evaluate_cvae_loss(model, prior, loss_fn, loader, step):
    if loader is None or len(loader) == 0:
        return {"loss": float("nan"), "ce_per_token": float("nan"), "tok_acc": float("nan")}
    model.eval()
    prior.eval()
    stats_list = []
    with torch.no_grad():
        for batch in loader:
            _, stats = _cvae_forward_loss(model, prior, loss_fn, batch, step, train=False)
            stats_list.append(stats)
    return _aggregate_cvae_stats(stats_list, len(loader))


def save_cvae_checkpoint(path, model, prior, loss_fn, epoch, val_metrics, split_tag):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "prior_state_dict": prior.state_dict(),
            "loss_state_dict": loss_fn.state_dict(),
            "epoch": int(epoch),
            "val_metrics": val_metrics,
            "split_tag": split_tag,
            "variant": VARIANT_NAME,
            "mode": mode,
            "latent_dim": latent_dim,
        },
        path,
    )


def load_cvae_checkpoint(path, model, prior, loss_fn):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    prior.load_state_dict(ckpt["prior_state_dict"])
    if "loss_state_dict" in ckpt:
        loss_fn.load_state_dict(ckpt["loss_state_dict"])
    return ckpt


def train_cvae_split(run_name, ctx, max_epochs=MAX_EPOCHS, patience=EARLY_STOP_PATIENCE, val_every=VAL_EVERY, min_delta=EARLY_STOP_MIN_DELTA):
    set_data_context(ctx)
    local_model, local_prior, local_loss_fn, local_optim, local_optim2 = init_model_bundle()
    best_val = float("inf")
    best_epoch = -1
    stale_checks = 0
    history = []
    ckpt_path = CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_{run_name}_cv_es_best.pth"
    progress = tqdm(range(max_epochs), desc=run_name)

    for epoch_idx in progress:
        train_metrics = train_one_epoch(
            local_model,
            local_prior,
            local_loss_fn,
            local_optim,
            local_optim2,
            train_dataloader,
            epoch_idx,
        )

        should_validate = len(val_dataloader) > 0 and ((epoch_idx + 1) % val_every == 0 or epoch_idx == max_epochs - 1)
        val_metrics = {"loss": float("nan"), "ce_per_token": float("nan"), "tok_acc": float("nan")}
        if should_validate:
            val_metrics = evaluate_cvae_loss(local_model, local_prior, local_loss_fn, val_dataloader, epoch_idx)
            val_loss = val_metrics["loss"]
            if val_loss < best_val - min_delta:
                best_val = val_loss
                best_epoch = epoch_idx + 1
                stale_checks = 0
                save_cvae_checkpoint(ckpt_path, local_model, local_prior, local_loss_fn, best_epoch, val_metrics, SPLIT_TAG)
            else:
                stale_checks += 1

            if patience is not None and stale_checks >= patience:
                progress.set_description(f"{run_name} early_stop epoch={epoch_idx+1} best_val={best_val:.6f}")
                break

        row = {
            "run": run_name,
            "epoch": epoch_idx + 1,
            "train_loss": train_metrics["loss"],
            "train_ce_per_token": train_metrics["ce_per_token"],
            "train_tok_acc": train_metrics["tok_acc"],
            "val_loss": val_metrics["loss"],
            "val_ce_per_token": val_metrics["ce_per_token"],
            "val_tok_acc": val_metrics["tok_acc"],
            "best_val_loss": best_val,
            "best_epoch": best_epoch,
            "split_tag": SPLIT_TAG,
        }
        history.append(row)
        progress.set_description(
            f"{run_name} train={train_metrics['loss']:.4f} val={val_metrics['loss']:.4f} best={best_val:.4f}"
        )

        with status_out:
            clear_output(wait=True)
            print(pd.DataFrame(history).tail(5).to_string(index=False))
            print("split:", SPLIT_TAG)
            print("leak_counts:", split_info["leak_counts"])

    if ckpt_path.exists():
        load_cvae_checkpoint(ckpt_path, local_model, local_prior, local_loss_fn)
    else:
        save_cvae_checkpoint(ckpt_path, local_model, local_prior, local_loss_fn, max_epochs, {"loss": float("nan")}, SPLIT_TAG)
        best_epoch = max_epochs

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_{run_name}_cv_es_history.csv", index=False)
    result = {
        "run": run_name,
        "best_val_loss": float(best_val),
        "best_epoch": int(best_epoch),
        "epochs_ran": int(hist_df["epoch"].max()) if not hist_df.empty else 0,
        "checkpoint": str(ckpt_path),
        "split_tag": SPLIT_TAG,
    }
    return local_model, local_prior, local_loss_fn, hist_df, result


cv_results = []
cv_histories = {}
if RUN_FULL_CV:
    for cv_val_fold in CV_VAL_FOLDS:
        ctx = build_split_context(
            test_fold=TEST_FOLD,
            val_fold=cv_val_fold,
            tag_suffix=f"cv_val{cv_val_fold}",
        )
        model_cv, prior_cv, loss_fn_cv, hist_df, result = train_cvae_split(
            run_name=f"cv_val{cv_val_fold}",
            ctx=ctx,
        )
        result["val_fold"] = int(cv_val_fold)
        cv_results.append(result)
        cv_histories[cv_val_fold] = hist_df

cv_results_df = pd.DataFrame(cv_results)
if not cv_results_df.empty:
    cv_results_df.to_csv(CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_cv_es_results.csv", index=False)
    display(cv_results_df)

if RUN_FINAL_TRAIN_ON_TRAINVAL:
    if not cv_results_df.empty and cv_results_df["best_epoch"].gt(0).any():
        FINAL_EPOCHS = int(np.median(cv_results_df.loc[cv_results_df["best_epoch"] > 0, "best_epoch"]))
    else:
        FINAL_EPOCHS = MAX_EPOCHS
    final_ctx = build_split_context(
        test_fold=TEST_FOLD,
        val_fold=None,
        tag_suffix="final_trainval",
    )
    model, prior, loss_fn, final_history, final_result = train_cvae_split(
        run_name="final_trainval",
        ctx=final_ctx,
        max_epochs=FINAL_EPOCHS,
        patience=None,
        val_every=FINAL_EPOCHS,
    )
    optim = None
    optim2 = None
    final_history.to_csv(CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_cv_es_final_trainval_history.csv", index=False)
    pd.DataFrame([final_result]).to_csv(CV_CHECKPOINT_DIR / f"{VARIANT_NAME}_cv_es_final_trainval_result.csv", index=False)
else:
    default_ctx = build_split_context(test_fold=TEST_FOLD, val_fold=VAL_FOLD, tag_suffix="single")
    model, prior, loss_fn, loss_history_df, single_result = train_cvae_split(run_name="single", ctx=default_ctx)

loss_arr = final_history["train_loss"].tolist() if RUN_FINAL_TRAIN_ON_TRAINVAL else loss_history_df["train_loss"].tolist()
print("Current split after training:")
print(split_summary.to_string(index=False))
print("leak_counts:", split_info["leak_counts"])


In [ ]:
if len(val_dataset) == 0:
    print("Current context has no validation split. Use cv_results_df for internal CV validation metrics.")
else:
    val_metrics = evaluate_cvae_loss(model, prior, loss_fn, val_dataloader, step=0)
    print("[VAL]", val_metrics)


In [ ]:
model.eval()
prior.eval()
results = []
origin = []

properties_results = []
properties_origin = []
EVAL_SPLIT_NAME = "test"
EVAL_DATASET = test_dataset
EVAL_DATALOADER = test_dataloader
print(EVAL_SPLIT_NAME, len(EVAL_DATASET))

with torch.no_grad():
    for smiles_enc, smiles_dec_input, smiles_dec_output, properties in EVAL_DATALOADER:
        smiles_enc = smiles_enc.to(device)
        smiles_dec_input = smiles_dec_input.to(device)
        smiles_dec_output = smiles_dec_output.to(device)
        properties = properties.to(device)

        enc_smi_mask = smiles_enc == PAD_IDX
        dec_smi_mask = smiles_dec_input == PAD_IDX

        result, tgt, means, log_var, z = model(
            smiles_enc,
            smiles_dec_input,
            properties,
            enc_smi_mask,
            dec_smi_mask,
        )

        results.append(result)
        origin.append(smiles_dec_output)

        properties_results.append(tgt)
        properties_origin.append(properties.squeeze(-1))

results = torch.cat(results, dim=0)
origin = torch.cat(origin, dim=0)
results = nn.functional.softmax(results, dim=-1)
argmax_indices = torch.argmax(results, dim=-1)
output = torch.nn.functional.one_hot(argmax_indices, num_classes=results.size(-1))
print(argmax_indices)
print(results.shape)
print(origin.shape)

from sklearn.metrics import mean_absolute_error
properties_origin = torch.cat(properties_origin, dim=0)
properties_results = torch.cat(properties_results, dim=0)
MAE_2 = mean_absolute_error(properties_origin.cpu(), properties_results.cpu())
print("MAE(properties) : ", MAE_2)

results_smiles = []
origin_smiles = []

for row in argmax_indices:
    smiles = tok_ids_to_smiles(row.tolist(), index_to_token)
    results_smiles.append(smiles or "")

for row in origin:
    smiles = tok_ids_to_smiles(row.tolist(), index_to_token)
    origin_smiles.append(smiles or "")


In [ ]:
origin_smiles = [smiles.removesuffix("EOS").strip() for smiles in origin_smiles]
results_smiles = [smiles.removesuffix("EOS").strip() for smiles in results_smiles]

for i in range(len(results_smiles)):
    if(origin_smiles[i] != results_smiles[i]):
        print(i, "번째 다름!")
    print("real smiles      : ", origin_smiles[i])
    print("predicted smiles : ", results_smiles[i])


MAE = mean_absolute_error(origin.cpu(), torch.argmax(results.cpu(), dim=-1))
print("MAE : ", MAE)



In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import DataStructs, rdFingerprintGenerator
RDLogger.DisableLog('rdApp.error')

generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def tanimoto_similarity(smiles1, smiles2):
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    fp1 = generator.GetFingerprint(mol1)
    fp2 = generator.GetFingerprint(mol2)
    return DataStructs.TanimotoSimilarity(fp1, fp2)

def is_valid(smiles):
    return Chem.MolFromSmiles(smiles) is not None

TS = 0.0
canbe = 0
notbe = 0

for sm, orig in zip(results_smiles, origin_smiles):
    if(len(sm)==0):
        notbe += 1
        continue
    try:
        sm = PS(sm).canonicalize.psmiles
        if is_valid(sm) and is_valid(orig):
            sim = tanimoto_similarity(sm, orig)
            TS += sim
            canbe += 1
        else:
            notbe += 1
    except:
        notbe += 1
        continue


if canbe > 0:
    print("Tanimoto Similarity : ", TS / canbe)
else:
    print("No valid molecules to compare.")

print("가능한 분자 개수 :", canbe)
print("불가능한 분자 개수 :", notbe)
print("Valid fraction      :", canbe / len(results_smiles))


In [ ]:
def save_weights():
    variant_name = os.environ.get("MODELS_VARIANT", mode)
    out_dir = PROJECT_ROOT / "checkpoints" / "cv_es" / variant_name
    out_dir.mkdir(parents=True, exist_ok=True)
    save_path = out_dir / f"model_weights_{variant_name}_cv_es.pth"
    torch.save(model.state_dict(), save_path)

    prior_path = out_dir / f"model_weights_prior_{variant_name}_cv_es.pth"
    torch.save(prior.state_dict(), prior_path)
    print("saved:", save_path)
    print("saved:", prior_path)


def load_weights(latent):
    from models.Trans import CVAE as TransCVAE, PriorNet as TransPriorNet
    from models.LSTM import LSTMCVAE, LSTMPriorNet

    variant_name = os.environ.get("MODELS_VARIANT", mode)
    out_dir = PROJECT_ROOT / "checkpoints" / "cv_es" / variant_name
    if mode == "Trans":
        model = TransCVAE(latent_dim=latent).to(device).eval()
        prior = TransPriorNet(y_dim=1, latent_dim=latent).to(device).eval()
    else:
        model = LSTMCVAE(latent_dim=latent).to(device).eval()
        prior = LSTMPriorNet(y_dim=1, latent_dim=latent).to(device).eval()

    model_path = out_dir / f"model_weights_{variant_name}_cv_es.pth"
    prior_path = out_dir / f"model_weights_prior_{variant_name}_cv_es.pth"
    model.load_state_dict(torch.load(model_path, map_location=device))
    prior.load_state_dict(torch.load(prior_path, map_location=device))
    return model, prior


In [ ]:
save_weights()
